# 📈 Stock / Commodity Forecast

Questo notebook scarica una serie temporale da Yahoo Finance e produce previsioni con un **pipeline di model selection**:

1. **Backtest** – ogni modello viene valutato su `N_WINDOWS` finestre consecutive (rolling origin) delle ultime osservazioni
2. **Selezione** – vince il modello con l'errore più basso (`SELECTION_METRIC`), oppure un **ensemble** dei migliori (`STRATEGY`)
3. **Retrain & Forecast** – il vincitore viene ritrainato su tutta la serie e produce la previsione finale

`MAE / Naive` < 1 significa che il modello batte il random walk («domani = oggi») sulle stesse finestre.

### Modelli disponibili
| Modello | Tipo |
|---|---|
| **Chronos-2**, **Chronos-Bolt**, **Chronos** | Amazon, foundation model zero-shot |
| **Timestral**, **TimesFM-3**, **TiRex** | Mistral, Google, NX-AI – foundation model zero-shot (extra opzionali) |
| **N-HiTS**, **N-BEATS** | NeuralForecast, addestrati al volo |
| **XGBoost** | Gradient boosting su lag |
| **Auto-ARIMA**, **AutoETS**, **Theta** | Statistici |
| **Naive**, **Drift** | Baseline (random walk) |

---
### Ticker di esempio
| Asset | Ticker |
|---|---|
| Apple | `AAPL` |
| S&P 500 | `^GSPC` |
| Oro | `GC=F` |
| Petrolio WTI | `CL=F` |
| Bitcoin | `BTC-USD` |
| EUR/USD | `EURUSD=X` |

Il catalogo completo è in `stock_forecast.data.TICKER_CATALOG`.

In [ ]:
import sys

sys.path.insert(0, "../src")

from stock_forecast.data import TICKER_CATALOG, download_series, get_ticker_info
from stock_forecast.models import available_models
from stock_forecast.pipeline import run_pipeline, select_best_model
from stock_forecast.plot import plot_backtest, plot_forecast, print_summary, print_validation_table

## 1 · Configurazione

In [ ]:
# ── Parametri principali ──────────────────────────────────────────────
TICKER = "CL=F"  # Yahoo Finance ticker   (es: AAPL, ^GSPC, GC=F, BTC-USD)
PERIOD = "max"  # storico scaricato       (6mo | 1y | 2y | 5y | 10y | max)
INTERVAL = "1d"  # granularità             (1d | 1wk | 1mo)
HORIZON = 15  # giorni / barre da prevedere

# ── Opzioni modelli ───────────────────────────────────────────────────
CANDIDATES = ["Chronos-2", "Timestral", "TimesFM-3", "TiRex", "Chronos-Bolt", "Theta", "Naive"]
MODELS = [m for m in CANDIDATES if m in available_models()]  # salta gli extra non installati
CHRONOS_SIZE = "large"  # tiny | mini | small | base | large  (solo per il modello "Chronos" originale)
MAX_STEPS = 500  # iterazioni di training per N-HiTS e N-BEATS
SELECTION_METRIC = "mae"  # metrica per la selezione (mae | rmse | mape | rel_mae)
N_WINDOWS = 3  # finestre di backtest
STRATEGY = "auto"  # best | ensemble | auto (ensemble solo se batte il miglior modello)

print(f"Ticker  : {TICKER}  |  Periodo: {PERIOD}  |  Orizzonte: {HORIZON} barre")
print(f"Modelli : {', '.join(MODELS)}")
print(f"Metrica : {SELECTION_METRIC}  |  Backtest: {N_WINDOWS} finestre  |  Strategia: {STRATEGY}")

## 2 · Download dati

In [ ]:
info = get_ticker_info(TICKER)
print(f"\nAsset   : {info.get('name', TICKER)}")
print(f"Ticker  : {info['ticker']}")
print(f"Currency: {info.get('currency', '—')}")
print(f"Sector  : {info.get('sector', '—')}")

df = download_series(TICKER, period=PERIOD, interval=INTERVAL)
print(f"\nSerie scaricata: {len(df)} osservazioni")
print(f"Da {df['ds'].min().date()}  →  {df['ds'].max().date()}")
df.tail()

## 3 · Model Selection Pipeline

Il pipeline:
1. Backtest di ogni modello su `N_WINDOWS` finestre da `HORIZON` osservazioni
2. Classifica per `SELECTION_METRIC` (e, se richiesto, ensemble dei migliori)
3. Retrain sul dataset completo → previsione finale

In [ ]:
val_results, best_forecast = run_pipeline(
    df,
    horizon=HORIZON,
    models=MODELS,
    chronos_size=CHRONOS_SIZE,
    max_steps=MAX_STEPS,
    selection_metric=SELECTION_METRIC,
    n_windows=N_WINDOWS,
    strategy=STRATEGY,
)

best_name = select_best_model(val_results, metric=SELECTION_METRIC)
print_validation_table(val_results, best_name=best_name)

print_summary({best_forecast.model_name: best_forecast}, ticker_info=info)

## 4 · Grafico interattivo

In [ ]:
fig = plot_forecast(
    best_forecast,
    ticker=info.get("name", TICKER),
    last_n_days=180,
    show_volume=True,
    show_members=True,  # se il vincitore è un ensemble, mostra anche i singoli modelli
)
fig.show()

plot_backtest(val_results, df).show()

## 5 · Salva grafico

In [ ]:
import os

os.makedirs("../outputs", exist_ok=True)
safe_ticker = TICKER.replace("=", "_").replace("^", "")
out_path = f"../outputs/{safe_ticker}_forecast.html"
fig.write_html(out_path)
print(f"Grafico salvato in: {out_path}")